# Projekt 01 (basic) — DQN von Grund auf: CartPole balancieren

> **Modul 14 — Deep Reinforcement Learning** · Format: **Jupyter Notebook (PyTorch)**
>
> **Warum dieses Format?** Deep RL lebt von *beobachten*: Lernkurven, die zappeln, ein Pol, der
> irgendwann stehen bleibt. Ein Notebook verbindet Code, Training und Visualisierung — ideal für
> den ersten eigenen **Deep-Q-Network**-Agenten.

## Ziel
Du baust ein **DQN** (Mnih et al. 2015 — das „Atari-Paper", hier im Kleinformat) und bringst
damit einen Agenten bei, die **CartPole**-Aufgabe zu lösen: einen Stab auf einem fahrbaren
Wagen durch Links/Rechts-Schübe **aufrecht** zu balancieren. Die Umgebung (die Physik) bauen
wir **selbst** — ganz ohne `gym`.

Du implementierst die **zwei konzeptuellen Kernstücke**; der Rest (Physik, Replay-Puffer,
Netz, Trainingsschleife, Plots) ist vorgegeben:
1. die **ε-greedy-Aktionswahl** (`select_action`),
2. das **DQN-Ziel & den Verlust** (`learn`) — das Herz des Algorithmus.

## Vorwissen
Skript Modul 14, Abschnitt **2** (DQN, Experience Replay, Target Network). Modul 13 (Q-Learning,
ε-greedy). Modul 05 (PyTorch: `nn.Module`, Adam, Backprop).

## Was am Ende funktionieren soll
Eine Lernkurve, die von ~15 auf mehrere Hundert Schritte steigt, und eine **greedy-Auswertung**,
bei der der Agent den Pol über die volle Episodenlänge (500 Schritte) hält. Training: ~20–40 s
auf der CPU.

> **Arbeitsweise:** Fülle die `# TODO`-Stellen selbst. Musterlösung in
> `loesung/dqn_cartpole_loesung.ipynb`.

In [ ]:
import math, random, time
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Reproduzierbarkeit
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cpu")   # kleine Netze -> CPU ist hier schnell genug (und reproduzierbar)
print("torch", torch.__version__, "| device:", device)

## 1 · Die Umgebung: CartPole von Hand

Der Zustand ist $s=[x,\dot x,\theta,\dot\theta]$ (Wagenposition & -geschwindigkeit,
Pol-Winkel & -Winkelgeschwindigkeit). Zwei Aktionen: **0 = Schub links**, **1 = Schub rechts**.
Belohnung **+1 pro Schritt**, in dem der Pol noch steht. Die Episode endet, wenn der Wagen zu
weit fährt ($|x|>2.4$), der Pol zu weit kippt ($|\theta|>12°$) oder 500 Schritte erreicht sind.
Die Dynamik ist die klassische (Euler-integrierte) Pol-auf-Wagen-Physik — vollständig
vorgegeben.

In [ ]:
class CartPole:
    def __init__(self):
        self.g=9.8; self.mc=1.0; self.mp=0.1; self.l=0.5; self.fmag=10.0; self.tau=0.02
        self.mt=self.mc+self.mp; self.pml=self.mp*self.l
        self.x_thr=2.4; self.th_thr=12*math.pi/180; self.max_steps=500
        self.n_states=4; self.n_actions=2

    def reset(self):
        self.s=np.random.uniform(-0.05,0.05,4); self.steps=0
        return self.s.copy()

    def step(self, a):
        x,xd,th,thd=self.s
        f=self.fmag if a==1 else -self.fmag
        ct=math.cos(th); st=math.sin(th)
        temp=(f+self.pml*thd*thd*st)/self.mt
        thacc=(self.g*st-ct*temp)/(self.l*(4/3-self.mp*ct*ct/self.mt))
        xacc=temp-self.pml*thacc*ct/self.mt
        x+=self.tau*xd; xd+=self.tau*xacc; th+=self.tau*thd; thd+=self.tau*thacc
        self.s=np.array([x,xd,th,thd]); self.steps+=1
        done=bool(abs(x)>self.x_thr or abs(th)>self.th_thr or self.steps>=self.max_steps)
        return self.s.copy(), 1.0, done

# kurze Demo: eine Zufallspolitik faellt schnell um
env=CartPole(); s=env.reset(); steps=0; done=False
while not done:
    s,r,done=env.step(random.randint(0,1)); steps+=1
print("Zufallspolitik haelt", steps, "Schritte (von max. 500)")

## 2 · Q-Netz und Replay-Puffer

**Q-Netz:** ein kleines MLP $s\;(4) \to 128 \to 128 \to Q(s,\cdot)\;(2)$ — ein Q-Wert je Aktion.
**Replay-Puffer:** ein Ringpuffer, der Transitionen $(s,a,r,s',\text{done})$ speichert; wir
trainieren auf **zufälligen Minibatches** daraus (entkorreliert die Daten). Beides vorgegeben.

In [ ]:
def make_qnet():
    return nn.Sequential(
        nn.Linear(4,128), nn.ReLU(),
        nn.Linear(128,128), nn.ReLU(),
        nn.Linear(128,2),
    ).to(device)

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buf=deque(maxlen=capacity)
    def push(self, s,a,r,s2,done):
        self.buf.append((s,a,r,s2,done))
    def sample(self, batch):
        bt=random.sample(self.buf, batch)
        s =torch.tensor(np.array([b[0] for b in bt]),dtype=torch.float32,device=device)
        a =torch.tensor([b[1] for b in bt],device=device).unsqueeze(1)
        r =torch.tensor([b[2] for b in bt],dtype=torch.float32,device=device).unsqueeze(1)
        s2=torch.tensor(np.array([b[3] for b in bt]),dtype=torch.float32,device=device)
        d =torch.tensor([float(b[4]) for b in bt],device=device).unsqueeze(1)
        return s,a,r,s2,d
    def __len__(self): return len(self.buf)

## 3 · Der DQN-Agent — **hier ist deine Arbeit**

Zwei Methoden sind auszufüllen:

**`select_action(state)` — ε-greedy** (wie in Modul 13, nur mit dem Netz statt der Tabelle):
mit Wahrscheinlichkeit $\varepsilon$ eine zufällige Aktion, sonst
$\arg\max_a Q(s,a;\theta)$. Tipp: fürs Netz `torch.tensor(state, dtype=torch.float32)` und
`with torch.no_grad():`.

**`learn()` — das DQN-Update.** Batch ist schon gezogen. Fülle **Ziel** und **Verlust**:
- **Ziel** (mit dem **Target-Netz** $\theta^-$, kein Gradient!):
  $$y = r + \gamma\,(1-\text{done})\,\max_{a'} Q(s',a';\theta^-).$$
- **Vorhersage:** $Q(s,a;\theta)$ — den Q-Wert der *tatsächlich gewählten* Aktion (`gather`).
- **Verlust:** `smooth_l1_loss`(Vorhersage, Ziel) (Huber — robuster als MSE).

Das **Target-Netz** wird danach per **Polyak-Mittel** sanft nachgezogen (vorgegeben) — das
stabilisiert das Lernen (Skript 2.1: „nicht auf ein bewegtes Ziel schießen").

In [ ]:
class DQNAgent:
    def __init__(self, gamma=0.99, lr=1e-3, batch=128, eps_start=1.0,
                 eps_min=0.02, eps_decay=0.99, tau=0.01):
        self.q  = make_qnet()
        self.qt = make_qnet(); self.qt.load_state_dict(self.q.state_dict())
        self.opt = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
        self.gamma=gamma; self.batch=batch; self.tau=tau
        self.eps=eps_start; self.eps_min=eps_min; self.eps_decay=eps_decay

    def select_action(self, state):
        # TODO: ε-greedy. Mit Wkt. self.eps eine zufaellige Aktion (0 oder 1),
        #       sonst argmax_a Q(state, a) aus self.q.
        #   with torch.no_grad():
        #       q = self.q(torch.tensor(state, dtype=torch.float32, device=device))
        #       return int(q.argmax())
        raise NotImplementedError

    def learn(self):
        if len(self.buffer) < 1000:      # erst lernen, wenn genug Erfahrung da ist
            return None
        s,a,r,s2,d = self.buffer.sample(self.batch)

        # Vorhersage Q(s,a;theta) fuer die tatsaechlich gewaehlte Aktion a:
        # TODO: q_sa = self.q(s).gather(1, a)
        q_sa = None  # TODO

        # Ziel y = r + gamma*(1-done)*max_a' Q(s',a';theta^-)   (Target-Netz, KEIN Gradient!)
        with torch.no_grad():
            # TODO: y = r + self.gamma * (1 - d) * self.qt(s2).max(1, keepdim=True)[0]
            y = None  # TODO

        # Huber-Verlust zwischen Vorhersage und Ziel:
        # TODO: loss = nn.functional.smooth_l1_loss(q_sa, y)
        loss = None  # TODO

        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()

        # Target-Netz sanft nachziehen (Polyak):  theta^- <- (1-tau) theta^- + tau theta
        with torch.no_grad():
            for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                pt.mul_(1 - self.tau).add_(self.tau * p)
        return float(loss)

    def decay_epsilon(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)


## 4 · Training

Die Schleife ist vorgegeben: pro Schritt Aktion wählen (ε-greedy), Umgebung schrittweise
fortschalten, Transition in den Puffer, **ein** Lernschritt. Nach jeder Episode ε senken.

Zwei Robustheits-Zutaten (Standard-Praxis, hier vorgegeben):
- **Best-Model-Checkpoint** — alle paar Episoden eine kurze **greedy**-Auswertung; die bisher
  beste Gewichts-Version wird gespeichert und am Ende zurückgeladen. So hängt das Ergebnis
  **nicht** davon ab, wo das (zappelige) Training gerade endet — DQN neigt zu *catastrophic
  forgetting* (siehe Fazit).
- **Solved-Kriterium** — erreicht die greedy-Auswertung ~500, ist die Aufgabe gelöst → Stopp.

> Wir setzen die Seeds direkt vor dem Training erneut, damit dieser Lauf **reproduzierbar** ist,
> unabhängig von den Zellen davor.

In [ ]:
import copy

def greedy_score(agent, env, episodes=5):
    saved=agent.eps; agent.eps=0.0
    res=[]
    for _ in range(episodes):
        s=env.reset(); done=False; total=0.0
        while not done:
            a=agent.select_action(s); s,r,done=env.step(a); total+=r
        res.append(total)
    agent.eps=saved
    return float(np.mean(res))

def train(agent, env, n_episodes=500, eval_every=10, solved=490.0, verbose=True):
    scores=[]; best=-1.0; best_weights=None; t0=time.time()
    for ep in range(n_episodes):
        s=env.reset(); done=False; total=0.0
        while not done:
            a=agent.select_action(s)
            s2,r,done=env.step(a)
            agent.buffer.push(s,a,r,s2,done)
            agent.learn()
            s=s2; total+=r
        agent.decay_epsilon()
        scores.append(total)
        if verbose and ep%20==0:
            print(f"ep {ep:3d}  score {total:5.0f}  avg20 {np.mean(scores[-20:]):6.1f}"
                  f"  eps {agent.eps:.2f}  t {time.time()-t0:.0f}s")
        # regelmaessige greedy-Auswertung -> bestes Modell merken, ggf. fruehzeitig stoppen
        if ep>=30 and ep%eval_every==0:
            g=greedy_score(agent, env, episodes=10)
            if g>best:
                best=g; best_weights=copy.deepcopy(agent.q.state_dict())
            if best>=solved:
                print(f"Geloest: greedy-Score {best:.0f} bei Episode {ep}.")
                break
    if best_weights is not None:
        agent.q.load_state_dict(best_weights)      # bestes Modell zurueckladen
    print(f"Training fertig in {time.time()-t0:.0f}s. Bester greedy-Score: {best:.0f}")
    return scores

# reproduzierbar machen (unabhaengig von den Zellen davor)
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
agent = DQNAgent()
scores = train(agent, CartPole(), n_episodes=500)

In [ ]:
def moving_average(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode="valid")

plt.figure(figsize=(8,4.2))
plt.plot(scores, alpha=0.3, label="Episoden-Score")
plt.plot(range(19, len(scores)), moving_average(scores,20), lw=2, label="gleitendes Mittel (20)")
plt.axhline(500, ls="--", color="gray", lw=1, label="Maximum (500)")
plt.xlabel("Episode"); plt.ylabel("Schritte balanciert (= Return)")
plt.title("DQN lernt CartPole"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5 · Greedy-Auswertung

Die Trainings-Scores enthalten noch ε-Exploration. Der wahre Test: die **greedy** Policy
(ε=0) über mehrere Episoden. Ein gut trainierter Agent hält den Pol die vollen **500** Schritte.

In [ ]:
def evaluate(agent, env, episodes=10):
    saved=agent.eps; agent.eps=0.0            # rein greedy
    results=[]
    for _ in range(episodes):
        s=env.reset(); done=False; total=0.0
        while not done:
            a=agent.select_action(s); s,r,done=env.step(a); total+=r
        results.append(total)
    agent.eps=saved
    return results

ev=evaluate(agent, CartPole(), episodes=10)
print("Greedy-Auswertung (10 Episoden):", ev)
print("Mittel:", np.mean(ev))

## 6 · Beobachtungen & Fazit

- Die Lernkurve ist **nicht monoton** — typisch für DQN. Oft steigt sie, **bricht ein**
  (*catastrophic forgetting*: das Netz „verlernt" kurzzeitig) und erholt sich wieder. Das ist
  ein direktes Symptom der **deadly triad** (Skript 1.2): Funktionsapproximation +
  Bootstrapping + off-policy. **Replay** und das **Target-Netz** zähmen sie so weit, dass der
  greedy-Agent am Ende zuverlässig die vollen 500 Schritte hält.
- Trotz zappeliger Trainingskurve ist die **greedy-Policy** am Ende meist perfekt — Trainings-
  Score (mit Exploration) und wahre Leistung (greedy) sind eben verschiedene Dinge.

### Mini-Aufgaben
1. **Ohne Target-Netz:** setze `tau=1.0` (Target = Online-Netz jeden Schritt). Wird das
   Training instabiler? (Das ist der Grund für das Target-Netz.)
2. **Double DQN** (Skript 2.2): ändere das Ziel zu
   $y=r+\gamma\,Q(s',\arg\max_{a'}Q(s',a';\theta);\theta^-)$ — Aktion aus dem Online-, Bewertung
   aus dem Target-Netz. Reduziert das die Überschätzung?
3. Kleineres Netz (z. B. $64$) oder anderes $\varepsilon$-Decay — wie robust ist das Lernen?
4. Plotte während des Trainings die mittleren **Q-Werte** — divergieren sie je?